In [1]:
# ===== CELL 1 : setup + paths =====
import os, glob, json, time, re, gc, warnings, subprocess, collections
from collections import Counter
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.abspath('./work')
os.makedirs(OUT, exist_ok=True)
INP = '/kaggle/input' if os.path.isdir('/kaggle/input') else os.path.abspath('./input')
COMP = '/kaggle/input/competitions/astroclimb'
LABELS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
COMBOS = ['image+text', 'text+text', 'image+image']
SEED = 0
T0 = time.time()
DEBUG = bool(int(os.environ.get('PLANE_DEBUG', '0')))   # local smoke test only; keep 0 on Kaggle

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except Exception:
    HAS_GPU = False
NCPU = os.cpu_count() or 2

def find(name, roots=None, isdir=False):
    for r in (roots or [INP, OUT]):
        for h in glob.glob(f'{r}/**/{name}', recursive=True):
            if os.path.isdir(h) == isdir:
                return h
    return None

def tlog(*a):
    print(f'[{(time.time()-T0)/60:5.1f} min]', *a, flush=True)

ESS = os.path.dirname(find('meta_train.parquet'))
IMGD = find('img384', isdir=True)
tlog('essentials:', ESS)
tlog('images    :', IMGD)
tlog('GPU:', HAS_GPU, '| CPUs:', NCPU, '| DEBUG:', DEBUG)

[  0.4 min] essentials: /kaggle/input/datasets/tahsanzahid/wasp-cache2026/essentials
[  0.4 min] images    : /kaggle/input/datasets/tahsanzahid/wasp-cache2026/img384
[  0.4 min] GPU: True | CPUs: 4 | DEBUG: False


In [2]:
# ===== CELL 2 : load cache (meta, hashes, texts, CLIP / SciNCL / DINOv2) + per-modality centering =====
def nrm(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)

def center(x):
    """per-modality centering: removes the modality's mean direction (anisotropy).
    Measured on held-out same_figure pairs: image->caption retrieval top-1 0.040 -> 0.075."""
    x = np.asarray(x, dtype=np.float32)
    return nrm(x - x.mean(0, keepdims=True))

mtr = pd.read_parquet(f'{ESS}/meta_train.parquet')
mte = pd.read_parquet(f'{ESS}/meta_test.parquet')
for d in (mtr, mte):
    for c in ['t1', 't2', 'h1', 'h2']:
        d[c] = d[c].astype(str)
    d['combo'] = np.where(d.t1 < d.t2, d.t1 + '+' + d.t2, d.t2 + '+' + d.t1)
mtr['y'] = mtr['label'].astype(str)

IH = pd.read_parquet(f'{ESS}/img_hashes.parquet').hash.astype(str).values
TH = pd.read_parquet(f'{ESS}/txt_hashes.parquet').hash.astype(str).values
II = {h: i for i, h in enumerate(IH)}
TI = {h: i for i, h in enumerate(TH)}

texts = pd.read_parquet(f'{ESS}/texts.parquet')
texts['hash'] = texts['hash'].astype(str)
TXT = texts.set_index('hash').loc[TH, 'text'].fillna('').astype(str).values

EMB = {
    'clip_i':  nrm(np.load(f'{ESS}/emb_clip_img.npy')),
    'clip_t':  nrm(np.load(f'{ESS}/emb_clip_txt.npy')),
    'sci_t':   center(np.load(f'{ESS}/emb_sci_txt.npy')),
    'dino_i':  center(np.load(f'{ESS}/emb_dino_img.npy')),
}
EMB['clip_ic'] = center(np.load(f'{ESS}/emb_clip_img.npy'))
EMB['clip_tc'] = center(np.load(f'{ESS}/emb_clip_txt.npy'))
assert EMB['clip_i'].shape[0] == len(IH) and EMB['clip_t'].shape[0] == len(TH)

ALL = pd.concat([mtr.assign(split='tr'), mte.assign(split='te')], ignore_index=True)
tlog('train', mtr.shape, '| test', mte.shape, '| texts', len(TH), '| images', len(IH))
print(mtr.groupby(['combo', 'y']).size().unstack(fill_value=0))

[  0.5 min] train (10000, 10) | test (10000, 8) | texts 14581 | images 14598
y            related_papers  same_figure  same_paper  unrelated_papers
combo                                                                 
image+image            1000            0        1000              1000
image+text             1000         1000        1000              1000
text+text              1000            0        1000              1000


In [3]:
# ===== CELL 3 : OCR text of every figure (tesseract, 2x upscale) =====
# If an ocr_384x2.parquet (hash, ocr) is attached as a dataset it is reused; otherwise computed here (~30-60 min CPU).
os.environ['OMP_THREAD_LIMIT'] = '1'

def _ocr_one(h):
    import pytesseract
    from PIL import Image
    try:
        im = Image.open(f'{IMGD}/{h}.jpg').convert('L')
        im = im.resize((im.width * 2, im.height * 2), Image.LANCZOS)
        return ' '.join(pytesseract.image_to_string(im, config='--psm 11').split())
    except Exception:
        return ''

def _have_tesseract():
    try:
        import pytesseract
        pytesseract.get_tesseract_version()
        return True
    except Exception:
        return False

OCR_OK = False
p = find('ocr_384x2.parquet')
if p is not None:
    o = pd.read_parquet(p)
    omap = dict(zip(o.hash.astype(str), o.ocr.fillna('').astype(str)))
    OCR = np.array([omap.get(h, '') for h in IH], dtype=object)
    OCR_OK = True
    tlog('OCR loaded from', p, '| missing:', sum(h not in omap for h in IH))
else:
    if not _have_tesseract():
        try:
            subprocess.run('pip install -q pytesseract', shell=True, timeout=300)
            subprocess.run('apt-get -qq update && apt-get -qq install -y tesseract-ocr', shell=True, timeout=900,
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as e:
            print('tesseract install failed:', e)
    if _have_tesseract():
        from multiprocessing import Pool
        hs = list(IH[:300]) if DEBUG else list(IH)
        res = []
        with Pool(NCPU) as pool:
            for i, r in enumerate(pool.imap(_ocr_one, hs, chunksize=16)):
                res.append(r)
                if i % 1000 == 0:
                    tlog(f'  OCR {i}/{len(hs)}')
        omap = dict(zip(hs, res))
        OCR = np.array([omap.get(h, '') for h in IH], dtype=object)
        pd.DataFrame({'hash': IH, 'ocr': OCR}).to_parquet(f'{OUT}/ocr_384x2.parquet')
        OCR_OK = True
        tlog('OCR computed and saved ->', f'{OUT}/ocr_384x2.parquet')
    else:
        OCR = np.array([''] * len(IH), dtype=object)
        print('WARNING: no tesseract and no OCR dataset -> OCR features disabled')

print('non-empty OCR:', np.mean([len(s) > 0 for s in OCR]).round(3))

[  0.5 min] OCR loaded from /kaggle/input/datasets/tonmoyy/ocr-384x2-parquet/ocr_384x2.parquet | missing: 0
non-empty OCR: 0.994


In [4]:
# ===== CELL 4 : figure "style profile" =====
# Figures from the same paper share plotting style (journal template, colormap, font, margins, size).
# Measured on image+image: cosine+length baseline 0.4235 -> with style pair features 0.4467 (+0.023).
def _style_one(h):
    from PIL import Image
    try:
        im = Image.open(f'{IMGD}/{h}.jpg')
        w, hh = im.size
        sz = os.path.getsize(f'{IMGD}/{h}.jpg')
        a = np.asarray(im.convert('RGB'), dtype=np.float32) / 255.
        g = a.mean(2)
        gx = np.abs(np.diff(g, axis=1)).mean(); gy = np.abs(np.diff(g, axis=0)).mean()
        sat = a.max(2) - a.min(2)
        q = (a[::3, ::3] * 5).astype(np.uint8)
        ncol = len(np.unique(q.reshape(-1, 3), axis=0))
        rw = (g > 0.95).mean(1); cw = (g > 0.95).mean(0)
        return [w, hh, w / hh, sz, sz / (w * hh), g.mean(), g.std(), (g > 0.95).mean(), (g < 0.2).mean(),
                sat.mean(), sat.std(), (sat > 0.15).mean(), gx, gy, gx / (gy + 1e-6), np.log1p(ncol),
                float((rw > 0.99).mean()), float((cw > 0.99).mean()),
                float(np.percentile(g, 5)), float(np.percentile(g, 95))]
    except Exception:
        return [0.] * 20

STYLE_COLS = ['w', 'h', 'aspect', 'bytes', 'bpp', 'gmean', 'gstd', 'white', 'dark', 'sat', 'satstd',
              'satfrac', 'gx', 'gy', 'gxy', 'ncol', 'rowwhite', 'colwhite', 'p5', 'p95']
STYLE_OK = False
try:
    p = find('style.parquet')
    if p is not None:
        s = pd.read_parquet(p)
        s['hash'] = s.hash.astype(str)
        STYLE = s.set_index('hash').reindex(IH)[STYLE_COLS].fillna(0).values.astype(np.float32)
        tlog('style loaded from', p)
    else:
        from multiprocessing import Pool
        with Pool(NCPU) as pool:
            R = pool.map(_style_one, list(IH), chunksize=64)
        STYLE = np.array(R, dtype=np.float32)
        pd.DataFrame(STYLE, columns=STYLE_COLS).assign(hash=IH).to_parquet(f'{OUT}/style.parquet')
        tlog('style computed ->', f'{OUT}/style.parquet')
    # robust scaling so tree splits and the LR model both behave
    med = np.median(STYLE, 0); iqr = (np.percentile(STYLE, 75, 0) - np.percentile(STYLE, 25, 0)).clip(1e-3)
    STYLE = ((STYLE - med) / iqr).astype(np.float32)
    STYLE_OK = np.isfinite(STYLE).all()
except Exception as e:
    print('style skipped:', repr(e)[:300])
    STYLE = np.zeros((len(IH), len(STYLE_COLS)), np.float32)
print('style ok:', STYLE_OK, STYLE.shape)

[  0.5 min] style loaded from /kaggle/input/datasets/tonmoyy/style1/style.parquet
style ok: True (14598, 20)


In [5]:
# ===== CELL 5 : external training data — the official full AstroCLIMB corpus on HuggingFace =====
# adsabs/AstroCLIMB: 94,233 figure rows over 10,000 papers, each with Paper DOI, caption,
# References DOIs and Citing DOIs. The competition labels are recoverable from that metadata:
#   same_paper = same DOI | related_papers = a citation edge | unrelated = no edge.
# So this corpus gives ~40x more supervision than the 10K Kaggle training pairs.
#
# Leakage check (run below, printed): 99.6% of Kaggle TRAIN captions appear in this corpus but
# 0.0% of Kaggle TEST captions do — the organisers held the test papers out of the public release.
# Any row matching a test caption is dropped anyway. No test object, and no test DOI, is ever used.
HF_REPO = 'adsabs/AstroCLIMB'
HF_SHARDS = 114
HF_PER_CLASS = 1500 if DEBUG else 40000     # sampled pairs per class
HF_HARD_FRAC = 0.4                           # share of `unrelated` drawn as hard negatives

def _norm(s):
    return re.sub(r'\s+', ' ', str(s)).strip()

HF_OK = False
try:
    p = find('hf_meta.parquet')
    if p is not None:
        hf = pd.read_parquet(p)
        tlog('HF metadata loaded from', p)
    else:
        import pyarrow.parquet as pq, fsspec
        from concurrent.futures import ThreadPoolExecutor
        COLS = ['UUID', 'Image ID', 'Paper DOI', 'Paper Title', 'Image Caption',
                'References DOIs', 'Citing DOIs']
        BASE = f'https://huggingface.co/datasets/{HF_REPO}/resolve/main/data/train-{{:05d}}-of-{HF_SHARDS:05d}.parquet'
        def _shard(i):
            for _ in range(3):
                try:
                    return pq.ParquetFile(fsspec.open(BASE.format(i)).open()).read(columns=COLS).to_pandas()
                except Exception:
                    time.sleep(3)
            return pd.DataFrame(columns=COLS)
        n = 4 if DEBUG else HF_SHARDS
        with ThreadPoolExecutor(8) as ex:
            hf = pd.concat(list(ex.map(_shard, range(n))), ignore_index=True)
        hf.to_parquet(f'{OUT}/hf_meta.parquet')
        tlog('HF metadata downloaded ->', f'{OUT}/hf_meta.parquet')

    hf['cap'] = [_norm(c) for c in hf['Image Caption']]
    hf = hf[hf.cap.str.len() >= 20].reset_index(drop=True)

    # ---- leakage audit + guard ----
    te_txt = set(mte.h1[mte.t1 == 'text']) | set(mte.h2[mte.t2 == 'text'])
    tr_txt = set(mtr.h1[mtr.t1 == 'text']) | set(mtr.h2[mtr.t2 == 'text'])
    capnorm = {h: _norm(t) for h, t in zip(TH, TXT)}
    hfset = set(hf.cap)
    in_tr = np.mean([capnorm[h] in hfset for h in tr_txt]) if tr_txt else 0
    in_te = np.mean([capnorm[h] in hfset for h in te_txt]) if te_txt else 0
    tlog(f'leakage audit — Kaggle captions found in HF corpus: train {in_tr:.3f} | test {in_te:.3f}')
    test_caps = {capnorm[h] for h in te_txt}
    bad_doi = set(hf.loc[hf.cap.isin(test_caps), 'Paper DOI'])
    if bad_doi:
        hf = hf[~hf['Paper DOI'].isin(bad_doi)].reset_index(drop=True)
    print(f'dropped {len(bad_doi)} DOI(s) that touch a test caption; HF rows kept: {len(hf)}')

    # ---- citation graph inside the corpus ----
    dois = set(hf['Paper DOI'])
    adj = collections.defaultdict(set)
    for doi, r, c in zip(hf['Paper DOI'], hf['References DOIs'], hf['Citing DOIs']):
        for x in (list(r) if r is not None else []):
            if x in dois:
                adj[doi].add(x); adj[x].add(doi)
        for x in (list(c) if c is not None else []):
            if x in dois:
                adj[doi].add(x); adj[x].add(doi)
    n_edge = sum(len(v) for v in adj.values()) // 2
    by = collections.defaultdict(list)
    for i, doi in enumerate(hf['Paper DOI']):
        by[doi].append(i)
    tlog(f'HF corpus: {len(hf)} figures | {len(dois)} papers | {n_edge} citation edges')

    # ---- sample training pairs (hard negatives: different papers that share title vocabulary) ----
    rng = np.random.default_rng(0)
    title_tok = {d: set(str(t).lower().split()) for d, t in zip(hf['Paper DOI'], hf['Paper Title'])}
    STOP = set('the a of and in for with from on to at by using its new an is are we'.split())
    inv = collections.defaultdict(list)
    for d, ts in title_tok.items():
        for t in ts:
            if len(t) > 4 and t not in STOP:
                inv[t].append(d)
    keys_sp = [k for k, v in by.items() if len(v) > 1]
    keys_rel = [k for k in adj if adj[k] and k in by]
    keys_all = list(by)

    def _pick(doi):
        v = by[doi]; return v[rng.integers(len(v))]

    pairs = []
    for _ in range(HF_PER_CLASS):
        v = by[keys_sp[rng.integers(len(keys_sp))]]
        i, j = rng.choice(len(v), 2, replace=False)
        pairs.append((v[i], v[j], 'same_paper'))
    while sum(1 for p in pairs if p[2] == 'related_papers') < HF_PER_CLASS:
        k = keys_rel[rng.integers(len(keys_rel))]
        nb = [x for x in adj[k] if x in by]
        if not nb:
            continue
        pairs.append((_pick(k), _pick(nb[rng.integers(len(nb))]), 'related_papers'))
    n_hard = int(HF_PER_CLASS * HF_HARD_FRAC)
    toks = [t for t, v in inv.items() if 2 <= len(set(v)) <= 400]
    got = 0
    while got < n_hard and toks:
        t = toks[rng.integers(len(toks))]
        cand = list(set(inv[t]))
        a, b = cand[rng.integers(len(cand))], cand[rng.integers(len(cand))]
        if a == b or b in adj[a]:
            continue
        pairs.append((_pick(a), _pick(b), 'unrelated_papers')); got += 1
    while got < HF_PER_CLASS:
        a, b = keys_all[rng.integers(len(keys_all))], keys_all[rng.integers(len(keys_all))]
        if a == b or b in adj[a]:
            continue
        pairs.append((_pick(a), _pick(b), 'unrelated_papers')); got += 1

    HFP = pd.DataFrame(pairs, columns=['a', 'b', 'label'])
    HFCAP = hf.cap.values.astype(object)

    # ---- pseudo-OCR view of every HF caption ----------------------------------------
    # The figure side of a Kaggle pair is represented by OCR text: a shuffled bag of partial
    # tokens, not a clean sentence. A model trained only on clean captions transfers badly to it
    # (measured: zero-shot image+image 0.32 vs text+text 0.62). So each HF caption also gets a
    # degraded twin, and a share of the training pairs uses it on one or both sides.
    _w = re.compile(r'[A-Za-z0-9][\w\-\.\+]*')
    def _pseudo_ocr(s, r):
        t = _w.findall(s)
        if not t:
            return ' '
        keep = [x for x in t if r.random() < 0.45]
        if len(keep) < 3:
            keep = t[:3]
        r.shuffle(keep)
        return ' '.join(keep)
    import random as _rnd
    _r = _rnd.Random(0)
    HFNOISE = np.array([_pseudo_ocr(c, _r) for c in HFCAP], dtype=object)
    u = rng.random(len(HFP))
    HFP['na'] = (u < 0.35) | ((u >= 0.5) & (u < 0.65))     # 35% both sides noisy, 15% side-a only
    HFP['nb'] = (u < 0.35)
    HF_OK = len(HFP) > 1000
    tlog('HF pairs:', HFP.label.value_counts().to_dict(),
         f'| hard negatives: {n_hard} | noisy-side pairs: {int((HFP.na | HFP.nb).sum())}')
except Exception as e:
    print('HF corpus skipped:', repr(e)[:400])
    HFCAP = np.array([], dtype=object); HFNOISE = np.array([], dtype=object)
    HFP = pd.DataFrame(columns=['a', 'b', 'label', 'na', 'nb'])
print('HF ok:', HF_OK)

[ 10.6 min] HF metadata downloaded -> /kaggle/working/hf_meta.parquet
[ 10.6 min] leakage audit — Kaggle captions found in HF corpus: train 0.818 | test 0.000
dropped 2 DOI(s) that touch a test caption; HF rows kept: 94059
[ 10.7 min] HF corpus: 94059 figures | 9998 papers | 26995 citation edges
[ 12.0 min] HF pairs: {'same_paper': 40000, 'related_papers': 40000, 'unrelated_papers': 40000} | hard negatives: 16000 | noisy-side pairs: 59911
HF ok: True


In [6]:
# ===== CELL 6 : (optional) VLM synthetic captions for figures =====
# Off by default: the HF corpus in cell 5 buys far more per GPU-hour than 3 hours of caption
# generation. Flip RUN_VLM to True (or attach a vlm_caption.parquet) if you want it; everything
# downstream picks it up automatically.
RUN_VLM = False
VLM_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
VLM_PROMPT = ('Describe this astronomy figure in one sentence: what quantity is plotted, '
              'what objects, surveys or instruments are named, and what the axes show.')
VLM_BATCH, VLM_MAXNEW = 8, 48
VLM_OK = False
VLM = np.array([''] * len(IH), dtype=object)
try:
    done = {}
    p = find('vlm_caption.parquet')
    if p is not None:
        v = pd.read_parquet(p)
        done = dict(zip(v.hash.astype(str), v.caption.fillna('').astype(str)))
        tlog(f'VLM captions loaded: {len(done)}')
    todo = [h for h in IH if not done.get(h)]
    if RUN_VLM and todo and HAS_GPU and not DEBUG:
        import torch
        from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
        from PIL import Image
        proc = AutoProcessor.from_pretrained(VLM_MODEL)
        proc.tokenizer.padding_side = 'left'
        vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            VLM_MODEL, torch_dtype=torch.float16, device_map='auto').eval()
        tmpl = proc.apply_chat_template(
            [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': VLM_PROMPT}]}],
            tokenize=False, add_generation_prompt=True)
        t0 = time.time()
        for i in range(0, len(todo), VLM_BATCH):
            chunk = todo[i:i + VLM_BATCH]
            ims = [Image.open(f'{IMGD}/{h}.jpg').convert('RGB') for h in chunk]
            batch = proc(text=[tmpl] * len(chunk), images=ims, return_tensors='pt', padding=True)
            batch = {k: (v.to(vlm.device) if hasattr(v, 'to') else v) for k, v in batch.items()}
            with torch.no_grad():
                out = vlm.generate(**batch, max_new_tokens=VLM_MAXNEW, do_sample=False)
            for h, txt in zip(chunk, proc.batch_decode(out[:, batch['input_ids'].shape[1]:],
                                                       skip_special_tokens=True)):
                done[h] = ' '.join(txt.split())
            if (i // VLM_BATCH) % 60 == 0:
                el = time.time() - t0
                tlog(f'  vlm {i+len(chunk)}/{len(todo)} {el/60:.1f} min '
                     f'eta {(len(todo)-i)/max((i+len(chunk))/max(el,1e-6),1e-6)/60:.0f} min')
                pd.DataFrame({'hash': list(done), 'caption': [done[h] for h in done]}).to_parquet(f'{OUT}/vlm_caption.parquet')
        pd.DataFrame({'hash': list(done), 'caption': [done[h] for h in done]}).to_parquet(f'{OUT}/vlm_caption.parquet')
        del vlm; gc.collect(); torch.cuda.empty_cache()
    if done:
        VLM = np.array([done.get(h, '') for h in IH], dtype=object)
        VLM_OK = float(np.mean([len(s) > 0 for s in VLM])) > 0.5
except Exception as e:
    print('VLM skipped:', repr(e)[:400])
print('VLM ok:', VLM_OK)

VLM ok: False


In [7]:
# ===== CELL 7 : encoders — SigLIP (figures+captions), Qwen3-Embedding & SciNCL over every text block =====
# Text blocks: 'cap' (Kaggle captions), 'ocr' (figure OCR), 'vlm' (figure VLM caption), 'hf' (HF corpus captions).
# Everything is cached as .npy in /kaggle/working and reused from an attached dataset on later runs.
try:
    import torch
    from torch.utils.data import Dataset, DataLoader
except Exception:
    torch = None
    Dataset = object

def _to_tensor(out):
    if torch.is_tensor(out):
        return out
    if getattr(out, 'pooler_output', None) is not None:
        return out.pooler_output
    return out.last_hidden_state[:, 0]

class _ImgDS(Dataset):
    def __init__(self, hashes, proc):
        self.h, self.p = hashes, proc
    def __len__(self):
        return len(self.h)
    def __getitem__(self, i):
        from PIL import Image
        im = Image.open(f'{IMGD}/{self.h[i]}.jpg').convert('RGB')
        return self.p(images=im, return_tensors='pt')['pixel_values'][0]

def load_cached(name):
    p = find(name)
    return np.load(p) if p is not None else None

# ---------- SigLIP so400m-384 ----------
try:
    si, st = load_cached('emb_siglip_img_ordered.npy'), load_cached('emb_siglip_txt_ordered.npy')
    if (si is None or st is None) and HAS_GPU and not DEBUG:
        from transformers import AutoModel, AutoProcessor
        M = 'google/siglip-so400m-patch14-384'
        sig = AutoModel.from_pretrained(M, torch_dtype=torch.float16).to('cuda').eval()
        sproc = AutoProcessor.from_pretrained(M)
        with torch.no_grad():
            outs = []
            dl = DataLoader(_ImgDS(list(IH), sproc.image_processor), batch_size=32,
                            num_workers=min(4, NCPU), pin_memory=True)
            for i, b in enumerate(dl):
                outs.append(_to_tensor(sig.get_image_features(pixel_values=b.to('cuda', torch.float16))).float().cpu().numpy())
                if i % 100 == 0:
                    tlog(f'  siglip img {i*32}/{len(IH)}')
            si = np.concatenate(outs)
            outs = []
            for i in range(0, len(TXT), 128):
                tk = sproc.tokenizer(list(TXT[i:i+128]), padding='max_length', truncation=True,
                                     max_length=64, return_tensors='pt').to('cuda')
                outs.append(_to_tensor(sig.get_text_features(**tk)).float().cpu().numpy())
            st = np.concatenate(outs)
        np.save(f'{OUT}/emb_siglip_img_ordered.npy', si); np.save(f'{OUT}/emb_siglip_txt_ordered.npy', st)
        del sig; gc.collect(); torch.cuda.empty_cache()
    if si is not None and st is not None:
        EMB['sig_i'], EMB['sig_t'] = nrm(si), nrm(st)
        EMB['sig_ic'], EMB['sig_tc'] = center(si), center(st)
        tlog('SigLIP ready', si.shape, st.shape)
except Exception as e:
    print('SigLIP skipped:', repr(e)[:300])

# ---------- text blocks ----------
BLOCKS = {'cap': list(TXT)}
if OCR_OK:
    BLOCKS['ocr'] = list(OCR)
if VLM_OK:
    BLOCKS['vlm'] = list(VLM)
if HF_OK:
    BLOCKS['hf'] = list(HFCAP)
    BLOCKS['hfn'] = list(HFNOISE)          # pseudo-OCR twins, so the model also learns noisy text
print('text blocks:', {k: len(v) for k, v in BLOCKS.items()})

def _batched(model, tok, strings, pool, bs, max_len, tag):
    torch.set_grad_enabled(False)
    order = np.argsort([len(s) for s in strings])
    outs = []
    for i in range(0, len(strings), bs):
        idx = order[i:i+bs]
        b = tok([strings[j] if strings[j] else ' ' for j in idx], padding=True, truncation=True,
                max_length=max_len, return_tensors='pt').to('cuda')
        h = model(**b).last_hidden_state
        h = h[:, -1] if pool == 'last' else h[:, 0]
        outs.append((idx, h.float().cpu().numpy()))
        if (i // bs) % 200 == 0:
            tlog(f'  {tag} {i}/{len(strings)}')
    E = np.zeros((len(strings), outs[0][1].shape[1]), np.float32)
    for idx, h in outs:
        E[idx] = h
    return E

TEXT_ENC = [('qwen', 'Qwen/Qwen3-Embedding-0.6B', 'last', 32, {'cap': 512, 'ocr': 256, 'vlm': 128, 'hf': 512, 'hfn': 256}),
            ('sci',  'malteos/scincl',             'cls',  64, {'cap': 256, 'ocr': 256, 'vlm': 128, 'hf': 256, 'hfn': 256})]
for short, repo, pool, bs, maxlens in TEXT_ENC:
    try:
        need, got = [], {}
        for blk in BLOCKS:
            if short == 'sci' and blk == 'cap':          # already in the cache as emb_sci_txt.npy
                continue
            e = load_cached(f'emb_{short}_{blk}.npy')
            need.append(blk) if e is None else got.update({blk: e})
        if need and HAS_GPU and not DEBUG:
            from transformers import AutoModel, AutoTokenizer
            tok = AutoTokenizer.from_pretrained(repo, padding_side='left' if pool == 'last' else 'right')
            mod = AutoModel.from_pretrained(repo, torch_dtype=torch.float16).to('cuda').eval()
            for blk in need:
                e = _batched(mod, tok, BLOCKS[blk], pool, bs, maxlens.get(blk, 256), f'{short}:{blk}')
                np.save(f'{OUT}/emb_{short}_{blk}.npy', e); got[blk] = e
                tlog(f'  {short}:{blk} done', e.shape)
            del mod; gc.collect(); torch.cuda.empty_cache()
        for blk, e in got.items():
            EMB[f'{short}_{blk}'] = center(e) if short == 'sci' else nrm(e)
            if short == 'qwen':
                EMB[f'{short}_{blk}c'] = center(e)
    except Exception as e:
        print(f'{short} skipped:', repr(e)[:300])

# names used by the rest of the notebook
if 'qwen_cap' in EMB:
    EMB['qwen_t'], EMB['qwen_tc'] = EMB['qwen_cap'], EMB['qwen_capc']
if 'qwen_ocr' in EMB:
    EMB['qwen_o'] = EMB['qwen_ocr']
if 'qwen_vlm' in EMB:
    EMB['qwen_v'] = EMB['qwen_vlm']
if 'sci_ocr' in EMB:
    EMB['sci_o'] = EMB['sci_ocr']
if 'sci_vlm' in EMB:
    EMB['sci_v'] = EMB['sci_vlm']
print('embedding spaces:', {k: v.shape[1] for k, v in sorted(EMB.items())})

[ 12.1 min] SigLIP ready (14598, 1152) (14581, 1152)
text blocks: {'cap': 14581, 'ocr': 14598, 'hf': 94059, 'hfn': 94059}


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: malteos/scincl
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ 57.6 min]   sci:ocr 0/14598
[ 57.9 min]   sci:ocr 12800/14598
[ 58.0 min]   sci:ocr done (14598, 768)
[ 58.0 min]   sci:hf 0/94059
[ 58.1 min]   sci:hf 12800/94059
[ 58.4 min]   sci:hf 25600/94059
[ 58.8 min]   sci:hf 38400/94059
[ 59.2 min]   sci:hf 51200/94059
[ 59.8 min]   sci:hf 64000/94059
[ 60.5 min]   sci:hf 76800/94059
[ 61.4 min]   sci:hf 89600/94059
[ 61.7 min]   sci:hf done (94059, 768)
[ 61.7 min]   sci:hfn 0/94059
[ 61.7 min]   sci:hfn 12800/94059
[ 61.8 min]   sci:hfn 25600/94059
[ 62.0 min]   sci:hfn 38400/94059
[ 62.1 min]   sci:hfn 51200/94059
[ 62.3 min]   sci:hfn 64000/94059
[ 62.6 min]   sci:hfn 76800/94059
[ 62.9 min]   sci:hfn 89600/94059
[ 63.1 min]   sci:hfn done (94059, 768)
embedding spaces: {'clip_i': 768, 'clip_ic': 768, 'clip_t': 768, 'clip_tc': 768, 'dino_i': 768, 'qwen_cap': 1024, 'qwen_capc': 1024, 'qwen_hf': 1024, 'qwen_hfc': 1024, 'qwen_hfn': 1024, 'qwen_hfnc': 1024, 'qwen_o': 1024, 'qwen_ocr': 1024, 'qwen_ocrc': 1024, 'qwen_t': 1024, 'qwen_tc': 1024

In [8]:
# ===== CELL 8 : one text-pair relation model, trained on the HF corpus =====
# All text (Kaggle captions, figure OCR, VLM captions, HF captions) goes into a single document pool,
# so the same feature function serves HF training pairs and every Kaggle subset:
#   text+text   -> caption  vs caption
#   image+text  -> caption  vs the figure's OCR / VLM text
#   image+image -> OCR/VLM  vs OCR/VLM
# The model never sees a Kaggle label, so its predictions on Kaggle rows are out-of-sample by
# construction and can be used both as features and as a blend arm.
GB = [b for b in ['cap', 'ocr', 'vlm', 'hf', 'hfn'] if b in BLOCKS]
GOFF, _o = {}, 0
for b in GB:
    GOFF[b] = _o; _o += len(BLOCKS[b])
GDOC = np.concatenate([np.asarray(BLOCKS[b], dtype=object) for b in GB])
NG = len(GDOC)
tlog('document pool:', {b: (GOFF[b], len(BLOCKS[b])) for b in GB}, '=', NG)

def _stack(prefix, alias):
    mats = []
    for b in GB:
        k = alias.get(b, f'{prefix}_{b}')
        if k not in EMB:
            return None
        mats.append(EMB[k])
    dims = {m.shape[1] for m in mats}
    return np.vstack(mats).astype(np.float32) if len(dims) == 1 else None

GSCI = _stack('sci', {'cap': 'sci_t'})
GQW = _stack('qwen', {'cap': 'qwen_cap'})
print('pool embeddings:', 'sci' if GSCI is not None else '-', 'qwen' if GQW is not None else '-')

# ---------- lexical index over the pool ----------
_mf = 120000 if DEBUG else 400000
gv_w = TfidfVectorizer(token_pattern=r'(?u)\b[\w\-\.\+]*\w\b', lowercase=False, sublinear_tf=True, max_df=0.5)
GW = gv_w.fit_transform(GDOC)
if DEBUG:                      # char n-grams are the slow part; skipped in the smoke test only
    from scipy.sparse import csr_matrix
    GC = csr_matrix((NG, 1), dtype=np.float32)
else:
    gv_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True, min_df=3, max_features=_mf)
    GC = gv_c.fit_transform(GDOC)
_gt = re.compile(r'[A-Za-z0-9][\w\-\.\+]*[A-Za-z0-9]|\d')
GTOK = [set(t for t in _gt.findall(s) if len(t) >= 2) for s in GDOC]
GDF = Counter(t for s in GTOK for t in s)
def _gidf(t):
    return np.log((NG + 1) / (GDF.get(t, 0) + 1))
GRARE = [{t for t in s if GDF.get(t, 0) <= 12 and len(t) >= 3} for s in GTOK]
GENT = [{t for t in s if (any(c.isdigit() for c in t) and any(c.isalpha() for c in t)) or (t.isupper() and len(t) >= 2)}
        for s in GTOK]
GNUM = [{t for t in s if re.fullmatch(r'\d+(\.\d+)?', t) and len(t) >= 2} for s in GTOK]
GLEN = np.log1p(np.array([len(s) for s in GDOC], np.float32))
tlog('pool lexical index built')

# hubness radius: mean cosine to a random anchor sample (CSLS without an all-pairs matrix)
rng_a = np.random.default_rng(0)
ANCH = rng_a.choice(NG, min(1024, NG), replace=False)
def _radius(M):
    r = np.zeros(len(M), np.float32)
    A = M[ANCH]
    for i in range(0, len(M), 4096):
        S = M[i:i+4096] @ A.T
        r[i:i+4096] = np.sort(S, 1)[:, -32:].mean(1)
    return r
RSCI = _radius(GSCI) if GSCI is not None else None
RQW = _radius(GQW) if GQW is not None else None

TP_COLS = None
def tp_feats(ia, ib):
    """text-pair features for global document indices"""
    global TP_COLS
    ia = np.asarray(ia); ib = np.asarray(ib)
    cols, F = [], []
    F += [np.asarray(GW[ia].multiply(GW[ib]).sum(1)).ravel(),
          np.asarray(GC[ia].multiply(GC[ib]).sum(1)).ravel()]
    cols += ['tfw', 'tfc']
    rn, rj, en, nu, ws, wj = ([] for _ in range(6))
    for a, b in zip(ia, ib):
        A, B = GRARE[a], GRARE[b]; I = A & B
        rn.append(len(I)); rj.append(len(I) / max(1, len(A | B)))
        en.append(len(GENT[a] & GENT[b])); nu.append(len(GNUM[a] & GNUM[b]))
        TA, TB = GTOK[a], GTOK[b]; I2 = TA & TB
        s = sum(_gidf(t) for t in I2)
        ws.append(s); wj.append(s / max(1e-6, sum(_gidf(t) for t in TA | TB)))
    F += [rn, rj, en, nu, ws, wj]
    cols += ['rare_n', 'rare_j', 'ent_n', 'num_n', 'idf_sum', 'idf_j']
    for nm, M, R in [('sci', GSCI, RSCI), ('qwen', GQW, RQW)]:
        if M is None:
            continue
        c = (M[ia] * M[ib]).sum(1)
        F += [c, 2 * c - R[ia] - R[ib], np.minimum(R[ia], R[ib]), np.maximum(R[ia], R[ib])]
        cols += [f'{nm}_cos', f'{nm}_csls', f'{nm}_rmin', f'{nm}_rmax']
    F += [np.minimum(GLEN[ia], GLEN[ib]), np.maximum(GLEN[ia], GLEN[ib])]
    cols += ['len_min', 'len_max']
    TP_COLS = cols
    return np.c_[tuple(np.asarray(f, np.float32) for f in F)].astype(np.float32)

TP3 = ['same_paper', 'related_papers', 'unrelated_papers']
HFPROB, HF_MODEL, HF_TRAIN = {}, None, None
try:
    if not HF_OK:
        raise RuntimeError('HF corpus unavailable')
    _ha = np.where(HFP.na.values, GOFF.get('hfn', GOFF['hf']), GOFF['hf']) + HFP.a.values
    _hb = np.where(HFP.nb.values, GOFF.get('hfn', GOFF['hf']), GOFF['hf']) + HFP.b.values
    Xh = tp_feats(_ha, _hb)
    yh = HFP.label.map({c: i for i, c in enumerate(TP3)}).values
    cut = int(0.9 * len(yh)); sh = np.random.default_rng(0).permutation(len(yh))
    tr_i, va_i = sh[:cut], sh[cut:]
    HF_MODEL = lgb.LGBMClassifier(objective='multiclass', num_class=3,
                                  n_estimators=200 if DEBUG else 900, learning_rate=0.05,
                                  num_leaves=63, colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                                  class_weight='balanced', verbose=-1, n_jobs=NCPU).fit(Xh[tr_i], yh[tr_i])
    tlog('HF pair model — held-out HF macro-F1:',
         round(f1_score(yh[va_i], HF_MODEL.predict(Xh[va_i]), average='macro'), 4))
    HF_TRAIN = (Xh, yh)

    # apply to every Kaggle pair, through the text views each subset has
    for combo in COMBOS:
        D = ALL[ALL.combo == combo].reset_index(drop=True)
        views = []
        if combo == 'text+text':
            a = np.array([GOFF['cap'] + TI[h] for h in D.h1]); b = np.array([GOFF['cap'] + TI[h] for h in D.h2])
            views.append(('cap', a, b))
        elif combo == 'image+text':
            sw = (D.t1 == 'image').values
            t = np.array([GOFF['cap'] + TI[h] for h in np.where(sw, D.h2, D.h1)])
            im = np.array([II[h] for h in np.where(sw, D.h1, D.h2)])
            if 'ocr' in GOFF:
                views.append(('ocr', t, GOFF['ocr'] + im))
            if 'vlm' in GOFF:
                views.append(('vlm', t, GOFF['vlm'] + im))
        else:
            a = np.array([II[h] for h in D.h1]); b = np.array([II[h] for h in D.h2])
            if 'ocr' in GOFF:
                views.append(('ocr', GOFF['ocr'] + a, GOFF['ocr'] + b))
            if 'vlm' in GOFF:
                views.append(('vlm', GOFF['vlm'] + a, GOFF['vlm'] + b))
        out = {}
        for nm, ia, ib in views:
            X = tp_feats(ia, ib)
            out[nm] = (X, HF_MODEL.predict_proba(X).astype(np.float32))
        HFPROB[combo] = out
        tlog(f'  HF probs for {combo}: views={list(out)}')
        # sanity: zero-shot quality on the Kaggle training rows of this subset
        m_ = (D.split == 'tr').values
        d = D[m_]
        if len(out) and set(d.y) <= set(TP3):
            P = sum(v[1] for v in out.values())[m_]
            tlog(f'    zero-shot HF->Kaggle {combo} macro-F1:',
                 round(f1_score(d.y.map({c: i for i, c in enumerate(TP3)}).values, P.argmax(1), average='macro'), 4))
except Exception as e:
    print('HF pair model skipped:', repr(e)[:400])

[ 63.1 min] document pool: {'cap': (0, 14581), 'ocr': (14581, 14598), 'hf': (29179, 94059), 'hfn': (123238, 94059)} = 217297
pool embeddings: sci qwen
[ 65.0 min] pool lexical index built
[ 65.6 min] HF pair model — held-out HF macro-F1: 0.5929
[ 65.7 min]   HF probs for image+text: views=['ocr']
[ 65.7 min]   HF probs for text+text: views=['cap']
[ 65.7 min]     zero-shot HF->Kaggle text+text macro-F1: 0.6242
[ 65.7 min]   HF probs for image+image: views=['ocr']
[ 65.7 min]     zero-shot HF->Kaggle image+image macro-F1: 0.4158


In [9]:
# ===== CELL 9 : Kaggle pair features (+ the HF model's probabilities as features) =====
# (a) similarity bundle per space: cosine, CSLS (hubness-corrected), log-rank in the full pool
#     — computed for raw AND per-modality-centered variants
# (b) lexical bundle over three text views: caption, OCR of the figure, VLM caption of the figure
# (c) style-profile pair features (image+image) — same paper => same plotting style
# (d) 2-hop proxies through the unlabeled pool
# (e) full |e1-e2|, e1*e2 blocks for the main spaces
NT, NI = len(TH), len(IH)

def topk_mean(Q, P, k, bs=2048):
    out = np.zeros(len(Q), np.float32)
    for i in range(0, len(Q), bs):
        S = Q[i:i+bs] @ P.T
        out[i:i+bs] = np.partition(S, -k, axis=1)[:, -k:].mean(1)
    return out

def rank_in_pool(Q, P, iq, ip, bs=2048):
    rk = np.zeros(len(iq), np.float32)
    for i in range(0, len(iq), bs):
        S = Q[iq[i:i+bs]] @ P.T
        s = S[np.arange(S.shape[0]), ip[i:i+bs]]
        rk[i:i+bs] = (S > s[:, None]).sum(1)
    return np.log1p(rk)

def best_match(Q, P, rP, k=5, bs=2048):
    out = np.zeros((len(Q), k), np.int64)
    for i in range(0, len(Q), bs):
        S = 2 * (Q[i:i+bs] @ P.T) - rP[None, :]
        out[i:i+bs] = np.argsort(-S, 1)[:, :k]
    return out

RAD = {}
def radius(a, b):
    if (a, b) not in RAD:
        RAD[(a, b)] = topk_mean(EMB[a], EMB[b], 11 if a == b else 10)
    return RAD[(a, b)]

XT, XI = ('sig_tc', 'sig_ic') if 'sig_tc' in EMB else ('clip_tc', 'clip_ic')
T2I = best_match(EMB[XT], EMB[XI], radius(XI, XT))
I2T = best_match(EMB[XI], EMB[XT], radius(XT, XI))
tlog('proxies built with', XT, XI)

# ---------- lexical index over caption | OCR | VLM caption ----------
DOCS = np.concatenate([TXT, OCR, VLM]).astype(str)       # text j -> j ; image i -> NT+i (OCR), NT+NI+i (VLM)
OCR_OFF, VLM_OFF = NT, NT + NI
tv_w = TfidfVectorizer(token_pattern=r'(?u)\b[\w\-\.\+]*\w\b', lowercase=False, sublinear_tf=True, max_df=0.5)
LW = tv_w.fit_transform(DOCS)
if DEBUG:
    from scipy.sparse import csr_matrix
    LC = csr_matrix((len(DOCS), 1), dtype=np.float32)
else:
    tv_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True, min_df=2, max_features=400000)
    LC = tv_c.fit_transform(DOCS)
_tok = re.compile(r'[A-Za-z0-9][\w\-\.\+]*[A-Za-z0-9]|\d')
TOKS = [set(t for t in _tok.findall(s) if len(t) >= 2 or t.isdigit()) for s in DOCS]
DF = Counter(t for s in TOKS[:NT] for t in s)
def _idf(t):
    return np.log((NT + 1) / (DF.get(t, 0) + 1))
RARE = [{t for t in s if DF.get(t, 0) <= 10 and len(t) >= 3} for s in TOKS]
ENT = [{t for t in s if (any(c.isdigit() for c in t) and any(c.isalpha() for c in t)) or (t.isupper() and len(t) >= 2)}
       for s in TOKS]
NUM = [{t for t in s if re.fullmatch(r'\d+(\.\d+)?', t) and len(t) >= 2} for s in TOKS]
LOWER = [{t.lower() for t in s if len(t) >= 4 and t.isalpha()} for s in TOKS]
tlog('lexical indices built over', len(DOCS), 'documents')

def lex(da, db, p):
    F = {f'{p}_tfw': np.asarray(LW[da].multiply(LW[db]).sum(1)).ravel(),
         f'{p}_tfc': np.asarray(LC[da].multiply(LC[db]).sum(1)).ravel()}
    rn, rj, en, nn_, ws, wj, lw = ([] for _ in range(7))
    for a, b in zip(da, db):
        A, B = RARE[a], RARE[b]; I = A & B
        rn.append(len(I)); rj.append(len(I) / max(1, len(A | B)))
        en.append(len(ENT[a] & ENT[b])); nn_.append(len(NUM[a] & NUM[b]))
        TA, TB = TOKS[a], TOKS[b]; I2 = TA & TB
        s_i = sum(_idf(t) for t in I2)
        ws.append(s_i); wj.append(s_i / max(1e-6, sum(_idf(t) for t in TA | TB)))
        lw.append(len(LOWER[a] & LOWER[b]))
    F.update({f'{p}_rare_n': rn, f'{p}_rare_j': rj, f'{p}_ent_n': en, f'{p}_num_n': nn_,
              f'{p}_idf_sum': ws, f'{p}_idf_j': wj, f'{p}_word_n': lw})
    return F

def sim_bundle(F, name, Ea, Eb, ia, ib, same_modality):
    A, B = EMB[Ea], EMB[Eb]
    c = (A[ia] * B[ib]).sum(1)
    ra, rb = radius(Ea, Eb)[ia], radius(Eb, Ea)[ib]
    F[f'{name}_cos'] = c
    F[f'{name}_csls'] = 2 * c - ra - rb
    k1 = rank_in_pool(A, B, ia, ib); k2 = rank_in_pool(B, A, ib, ia)
    if same_modality:
        F[f'{name}_rmin'], F[f'{name}_rmax'] = np.minimum(k1, k2), np.maximum(k1, k2)
        F[f'{name}_hubmin'], F[f'{name}_hubmax'] = np.minimum(ra, rb), np.maximum(ra, rb)
    else:
        F[f'{name}_r12'], F[f'{name}_r21'] = k1, k2
        F[f'{name}_hub1'], F[f'{name}_hub2'] = ra, rb

def sym(F, name, x, y):
    F[f'{name}_min'], F[f'{name}_max'] = np.minimum(x, y), np.maximum(x, y)

def pair_index(D):
    combo = D.combo.iloc[0]
    if combo == 'image+text':
        sw = (D.t1 == 'image').values
        a = np.array([TI[h] for h in np.where(sw, D.h2, D.h1)])
        b = np.array([II[h] for h in np.where(sw, D.h1, D.h2)])
        la = np.where(sw, D.len2, D.len1); lb = np.where(sw, D.len1, D.len2)
    elif combo == 'text+text':
        a = np.array([TI[h] for h in D.h1]); b = np.array([TI[h] for h in D.h2]); la, lb = D.len1.values, D.len2.values
    else:
        a = np.array([II[h] for h in D.h1]); b = np.array([II[h] for h in D.h2]); la, lb = D.len1.values, D.len2.values
    return a, b, np.log1p(la.astype(float)), np.log1p(lb.astype(float))

CROSS = [('clip_t', 'clip_i'), ('clip_tc', 'clip_ic'), ('sig_t', 'sig_i'), ('sig_tc', 'sig_ic')]
TSPACES = ['clip_t', 'clip_tc', 'sci_t', 'sig_t', 'sig_tc', 'qwen_t', 'qwen_tc']
ISPACES = ['clip_i', 'clip_ic', 'dino_i', 'sig_i', 'sig_ic']
FIG_TEXT = ['sci_o', 'qwen_o', 'sci_v', 'qwen_v']        # figure represented through OCR / VLM caption

def build_scalar(D):
    combo = D.combo.iloc[0]; a, b, la, lb = pair_index(D); F = {}
    if combo == 'image+text':
        for tsp, isp in CROSS:
            if tsp in EMB and isp in EMB:
                sim_bundle(F, tsp, tsp, isp, a, b, False)
        # caption (text space) vs the figure's own text views
        for cap_sp, fig_sp in [('sci_t', 'sci_o'), ('qwen_t', 'qwen_o'), ('sci_t', 'sci_v'), ('qwen_t', 'qwen_v')]:
            if cap_sp in EMB and fig_sp in EMB:
                sim_bundle(F, f'{cap_sp}__{fig_sp}', cap_sp, fig_sp, a, b, False)
        if OCR_OK:
            F.update(lex(a, OCR_OFF + b, 'cap_ocr'))
            F['ocr_len'] = np.log1p([len(OCR[i]) for i in b])
        if VLM_OK:
            F.update(lex(a, VLM_OFF + b, 'cap_vlm'))
            F['vlm_len'] = np.log1p([len(VLM[i]) for i in b])
        F['px_dino'] = (EMB['dino_i'][T2I[a, 0]] * EMB['dino_i'][b]).sum(1)
        F['px_dino5'] = (nrm(EMB['dino_i'][T2I[a]].mean(1)) * EMB['dino_i'][b]).sum(1)
        F['px_sci'] = (EMB['sci_t'][I2T[b, 0]] * EMB['sci_t'][a]).sum(1)
        F['px_sci5'] = (nrm(EMB['sci_t'][I2T[b]].mean(1)) * EMB['sci_t'][a]).sum(1)
        F.update(lex(I2T[b, 0], a, 'px_cap'))
        F['self_t'] = (T2I[a] == b[:, None]).any(1).astype(np.float32)
        F['self_i'] = (I2T[b] == a[:, None]).any(1).astype(np.float32)
        F['len_t'], F['len_i'] = la, lb
        if STYLE_OK:
            for j, c in enumerate(STYLE_COLS):
                F[f'sty_{c}'] = STYLE[b, j]
    elif combo == 'text+text':
        for sp in TSPACES:
            if sp in EMB:
                sim_bundle(F, sp, sp, sp, a, b, True)
        F.update(lex(a, b, 'cap'))
        F['px_dino'] = (EMB['dino_i'][T2I[a, 0]] * EMB['dino_i'][T2I[b, 0]]).sum(1)
        F['px_dino5'] = (nrm(EMB['dino_i'][T2I[a]].mean(1)) * nrm(EMB['dino_i'][T2I[b]].mean(1))).sum(1)
        sym(F, 'len', la, lb)
    else:
        for sp in ISPACES + FIG_TEXT:
            if sp in EMB:
                sim_bundle(F, sp, sp, sp, a, b, True)
        if OCR_OK:
            F.update(lex(OCR_OFF + a, OCR_OFF + b, 'ocr'))
            sym(F, 'ocr_len', np.log1p([len(OCR[i]) for i in a]), np.log1p([len(OCR[i]) for i in b]))
        if VLM_OK:
            F.update(lex(VLM_OFF + a, VLM_OFF + b, 'vlm'))
        F['px_sci'] = (EMB['sci_t'][I2T[a, 0]] * EMB['sci_t'][I2T[b, 0]]).sum(1)
        F['px_sci5'] = (nrm(EMB['sci_t'][I2T[a]].mean(1)) * nrm(EMB['sci_t'][I2T[b]].mean(1))).sum(1)
        F.update(lex(I2T[a, 0], I2T[b, 0], 'px_cap'))
        sym(F, 'len', la, lb)
        if STYLE_OK:                       # same paper => same plotting style (+0.023 measured)
            x, y = STYLE[a], STYLE[b]
            for j, c in enumerate(STYLE_COLS):
                F[f'sty_d_{c}'] = np.abs(x[:, j] - y[:, j])
                F[f'sty_mn_{c}'] = np.minimum(x[:, j], y[:, j])
                F[f'sty_mx_{c}'] = np.maximum(x[:, j], y[:, j])
    return pd.DataFrame({k: np.asarray(v, dtype=np.float32) for k, v in F.items()})

FULL_SPACES = {'image+text': [('clip_t', 'clip_i'), ('clip_tc', 'clip_ic'), ('sig_t', 'sig_i'),
                              ('sci_t', 'sci_v'), ('qwen_t', 'qwen_v')],
               'text+text': [('clip_t', 'clip_t'), ('sci_t', 'sci_t'), ('qwen_t', 'qwen_t')],
               'image+image': [('clip_i', 'clip_i'), ('dino_i', 'dino_i'), ('sig_i', 'sig_i'),
                               ('sci_v', 'sci_v'), ('sci_o', 'sci_o')]}

def build_full(D):
    combo = D.combo.iloc[0]; a, b, _, _ = pair_index(D); B = []
    for sa, sb in FULL_SPACES[combo]:
        if sa in EMB and sb in EMB:
            e1, e2 = EMB[sa][a], EMB[sb][b]
            B += [np.abs(e1 - e2), e1 * e2]
    return np.hstack(B).astype(np.float32)

DATA = {}
for combo in COMBOS:
    D = ALL[ALL.combo == combo].reset_index(drop=True)
    S = build_scalar(D); Fu = build_full(D)
    for nm, (Xv, Pv) in HFPROB.get(combo, {}).items():          # HF-trained relation model
        for j, c in enumerate(TP3):
            S[f'hf_{nm}_{c}'] = Pv[:, j]
        S[f'hf_{nm}_margin'] = Pv.max(1) - np.sort(Pv, 1)[:, -2]
    S = S.replace([np.inf, -np.inf], 0).fillna(0)
    Fu = np.nan_to_num(Fu, nan=0.0, posinf=0.0, neginf=0.0)
    DATA[combo] = (D, S, Fu)
    tlog(f'{combo}: rows={len(D)} scalar={S.shape[1]} full={Fu.shape[1]}')
    assert np.isfinite(S.values).all() and np.isfinite(Fu).all()

[ 66.1 min] proxies built with sig_tc sig_ic
[ 66.4 min] lexical indices built over 43777 documents
[ 67.1 min] image+text: rows=8000 scalar=87 full=5376
[ 67.6 min] text+text: rows=6000 scalar=59 full=5120
[ 68.1 min] image+image: rows=6000 scalar=130 full=6912


In [10]:
# ===== CELL 10 : per-subset models (same protocol as the team) + blend + per-class thresholds =====
FOLDS = StratifiedKFold(5, shuffle=True, random_state=0)

def lgb_params(K, **kw):
    p = dict(objective='multiclass', num_class=K, n_estimators=700, learning_rate=0.05, num_leaves=31,
             colsample_bytree=0.3, subsample=0.8, subsample_freq=1, class_weight='balanced',
             min_child_samples=20, reg_lambda=1.0, verbose=-1, n_jobs=NCPU)
    p.update(kw)
    if DEBUG:
        p['n_estimators'] = min(p['n_estimators'], 60)
    return p

def run_lgb(X, y, Xt, K, seeds=(0,), **kw):
    oof = np.zeros((len(y), K)); te = np.zeros((len(Xt), K))
    for s in seeds:
        for tr_i, va_i in FOLDS.split(X, y):
            m = lgb.LGBMClassifier(**lgb_params(K, random_state=s, **kw)).fit(X[tr_i], y[tr_i])
            oof[va_i] += m.predict_proba(X[va_i]) / len(seeds)
            te += m.predict_proba(Xt) / (5 * len(seeds))
    return oof, te

def run_lr(X, y, Xt, K, C=0.05):
    oof = np.zeros((len(y), K)); te = np.zeros((len(Xt), K))
    for tr_i, va_i in FOLDS.split(X, y):
        sc = StandardScaler().fit(X[tr_i])
        m = LogisticRegression(C=C, max_iter=300 if not DEBUG else 50, class_weight='balanced')
        m.fit(sc.transform(X[tr_i]), y[tr_i])
        oof[va_i] = m.predict_proba(sc.transform(X[va_i]))
        te += m.predict_proba(sc.transform(Xt)) / 5
    return oof, te

def mf1(y, P, w=None):
    return f1_score(y, (P * (1 if w is None else w)).argmax(1), average='macro')

def blend_weights(oofs, y, step=0.1):
    names = list(oofs); best = (-1, None)
    grid = np.arange(0, 1 + 1e-9, step)
    def rec(i, left, cur):
        nonlocal best
        if i == len(names) - 1:
            w = cur + [left]
            s = mf1(y, sum(wi * oofs[n] for wi, n in zip(w, names)))
            if s > best[0]:
                best = (s, {n: round(float(x), 2) for n, x in zip(names, w)})
            return
        for g in grid:
            if g <= left + 1e-9:
                rec(i + 1, left - g, cur + [g])
    rec(0, 1.0, [])
    return best

def class_mult(P, y, rounds=10):
    w = np.ones(P.shape[1]); best = mf1(y, P, w)
    for _ in range(rounds):
        moved = False
        for j in range(P.shape[1]):
            for m in (0.7, 0.8, 0.9, 0.95, 1.05, 1.1, 1.25, 1.4):
                w2 = w.copy(); w2[j] *= m
                s = mf1(y, P, w2)
                if s > best + 1e-5:
                    best, w, moved = s, w2, True
        if not moved:
            break
    return w, best

REPORT, SIZES, TESTP = {}, {}, {}

for combo in COMBOS:
    D, S, Fu = DATA[combo]
    m_ = (D.split == 'tr').values
    d, dt = D[m_].reset_index(drop=True), D[~m_].reset_index(drop=True)
    classes = [c for c in LABELS if c in set(d.y)]
    K = len(classes)
    y = d.y.map({c: i for i, c in enumerate(classes)}).values
    Xs, Xst = S.values[m_], S.values[~m_]
    Xa, Xat = np.hstack([Xs, Fu[m_]]), np.hstack([Xst, Fu[~m_]])
    tlog(f'--- {combo}: n={len(y)} K={K} scalar={Xs.shape[1]} all={Xa.shape[1]}')

    oofs, tests = {}, {}
    oofs['lgb_scalar'], tests['lgb_scalar'] = run_lgb(Xs, y, Xst, K, seeds=(0, 1, 2), n_estimators=600,
                                                      learning_rate=0.03, num_leaves=15, colsample_bytree=0.7)
    tlog(f'  lgb_scalar   {mf1(y, oofs["lgb_scalar"]):.4f}')
    oofs['lgb_all'], tests['lgb_all'] = run_lgb(Xa, y, Xat, K, seeds=(0,))
    tlog(f'  lgb_all      {mf1(y, oofs["lgb_all"]):.4f}')
    if not DEBUG:
        oofs['lr_all'], tests['lr_all'] = run_lr(Xa, y, Xat, K)
        tlog(f'  lr_all       {mf1(y, oofs["lr_all"]):.4f}')

    # HF-augmented arm: the text-pair model retrained per fold on HF pairs + this fold's Kaggle rows.
    # Only for the 3-class subsets, where the Kaggle label space equals the HF one.
    if HF_TRAIN is not None and classes == TP3 and combo in HFPROB and HFPROB[combo]:
        Xh_, yh_ = HF_TRAIN
        view = list(HFPROB[combo])[0]
        Xtp = HFPROB[combo][view][0]
        Xtp_tr, Xtp_te = Xtp[m_], Xtp[~m_]
        oof_h = np.zeros((len(y), K)); te_h = np.zeros((len(Xtp_te), K))
        for tr_i, va_i in FOLDS.split(Xtp_tr, y):
            g = lgb.LGBMClassifier(**lgb_params(K, n_estimators=200 if DEBUG else 900, num_leaves=63,
                                                colsample_bytree=0.8, random_state=0))
            g.fit(np.vstack([Xh_, Xtp_tr[tr_i]]), np.concatenate([yh_, y[tr_i]]))
            oof_h[va_i] = g.predict_proba(Xtp_tr[va_i]); te_h += g.predict_proba(Xtp_te) / 5
        oofs['hf_aug'], tests['hf_aug'] = oof_h, te_h
        tlog(f'  hf_aug       {mf1(y, oof_h):.4f}')

    s_bl, wts = blend_weights(oofs, y)
    P = sum(wts[n] * oofs[n] for n in oofs); Pt = sum(wts[n] * tests[n] for n in tests)
    mult, s_fin = class_mult(P, y)
    tlog(f'  blend {wts} -> {s_bl:.4f} | +class thresholds -> {s_fin:.4f}')
    print(classification_report(y, (P * mult).argmax(1), target_names=classes, digits=3))
    print(pd.DataFrame(confusion_matrix(y, (P * mult).argmax(1)), index=classes, columns=classes))

    REPORT[combo] = {'features_scalar': int(Xs.shape[1]), 'features_all': int(Xa.shape[1]),
                     **{n: round(mf1(y, o), 4) for n, o in oofs.items()},
                     'blend_weights': wts, 'blend': round(s_bl, 4), 'final': round(s_fin, 4)}
    SIZES[combo] = len(y)
    TESTP[combo] = (dt[['id', 'h1', 'h2']].copy(), classes, (Pt * mult))

    tag = combo.replace('+', '_')
    for n in oofs:
        np.save(f'{OUT}/D_oof_{n}_{tag}.npy', oofs[n].astype(np.float32))
        np.save(f'{OUT}/D_test_{n}_{tag}.npy', tests[n].astype(np.float32))
    np.save(f'{OUT}/D_oof_blend_{tag}.npy', (P * mult).astype(np.float32))
    np.save(f'{OUT}/D_test_blend_{tag}.npy', (Pt * mult).astype(np.float32))
    json.dump(classes, open(f'{OUT}/D_classes_{tag}.json', 'w'))

tot = sum(SIZES.values())
REPORT['overall_oof_macro_f1'] = round(sum(REPORT[c]['final'] * SIZES[c] for c in SIZES) / tot, 4)
tlog('OVERALL OOF macro-F1 =', REPORT['overall_oof_macro_f1'])

[ 68.1 min] --- image+text: n=4000 K=4 scalar=87 all=5463
[ 69.4 min]   lgb_scalar   0.5153
[ 84.9 min]   lgb_all      0.5007
[ 85.6 min]   lr_all       0.4329
[ 85.6 min]   blend {'lgb_scalar': 0.6, 'lgb_all': 0.3, 'lr_all': 0.1} -> 0.5238 | +class thresholds -> 0.5303
                  precision    recall  f1-score   support

     same_figure      0.767     0.693     0.728      1000
      same_paper      0.461     0.372     0.412      1000
  related_papers      0.397     0.363     0.379      1000
unrelated_papers      0.520     0.715     0.602      1000

        accuracy                          0.536      4000
       macro avg      0.536     0.536     0.530      4000
    weighted avg      0.536     0.536     0.530      4000

                  same_figure  same_paper  related_papers  unrelated_papers
same_figure               693         157             102                48
same_paper                139         372             250               239
related_papers             60     

In [11]:
# ===== CELL 11 : transductive paper-cluster consistency (test-time only, uses NO labels) =====
# Structural fact, verified on both splits: `related`/`unrelated` is a property of the PAPER PAIR,
# not of the object pair. Cluster objects into papers via confident same_paper / same_figure edges;
# every test pair between the same two clusters must then carry the same label.
#   train: 2 multi-edge cluster-pairs, 0 inconsistent | test: 227 multi-edge cluster-pairs, 0 inconsistent
# So predictions over such a group can be pooled. This only reorders probabilities — nothing is forced.
MERGE_TH = 0.55          # P(same_paper)+P(same_figure) needed to treat an edge as "same paper"

PROB = np.zeros((len(mte), len(LABELS)), np.float32)
rowof = pd.Index(mte.id).get_indexer
for combo in COMBOS:
    dt, classes, Pt = TESTP[combo]
    pos = rowof(dt.id.values)
    assert (pos >= 0).all()
    for j, c in enumerate(classes):
        PROB[pos, LABELS.index(c)] = Pt[:, j]
PROB = PROB / PROB.sum(1, keepdims=True).clip(1e-9)

H1, H2 = mte.h1.values, mte.h2.values
same_paper_p = PROB[:, LABELS.index('same_figure')] + PROB[:, LABELS.index('same_paper')]

parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

for h in np.concatenate([H1, H2]):
    find(h)
for a, b, s in zip(H1, H2, same_paper_p):
    if s >= MERGE_TH:
        union(a, b)

groups = collections.defaultdict(list)
for i, (a, b) in enumerate(zip(H1, H2)):
    ca, cb = find(a), find(b)
    groups[(min(ca, cb), max(ca, cb), ca == cb)].append(i)

n_cluster = len(set(find(h) for h in parent))
multi = {k: v for k, v in groups.items() if len(v) > 1 and not k[2]}
within = [i for k, v in groups.items() if k[2] for i in v]
tlog(f'paper clusters: {n_cluster} | cross-cluster groups with >1 test pair: {len(multi)} '
     f'({sum(len(v) for v in multi.values())} pairs) | pairs inside one cluster: {len(within)}')

PROB_C = PROB.copy()
# (a) pool evidence over every test pair connecting the same two papers (geometric mean = log-average)
for k, idx in multi.items():
    lp = np.log(PROB[idx].clip(1e-9)).mean(0)
    p = np.exp(lp - lp.max()); PROB_C[idx] = p / p.sum()
# (b) a pair whose two objects landed in the SAME paper cluster cannot be related/unrelated
PROB_C2 = PROB_C.copy()
if within:
    keep = [LABELS.index('same_figure'), LABELS.index('same_paper')]
    for i in within:
        v = PROB_C2[i].copy()
        v[[j for j in range(len(LABELS)) if j not in keep]] *= 0.25
        PROB_C2[i] = v / v.sum()

def to_sub(P):
    s = pd.DataFrame({'id': mte.id.values})
    for c in LABELS:
        s[c] = 0
    # a structural constraint that always holds: same_figure only exists in image+text pairs
    P = P.copy()
    not_it = (mte.combo != 'image+text').values
    P[not_it, LABELS.index('same_figure')] = 0
    pred = P.argmax(1)
    for j, c in enumerate(LABELS):
        s.loc[pred == j, c] = 1
    assert (s[LABELS].sum(1) == 1).all() and len(s) == len(mte)
    return s[['id'] + LABELS]

SUBS = {'base': to_sub(PROB), 'consistency': to_sub(PROB_C), 'consistency_within': to_sub(PROB_C2)}
for n, s in SUBS.items():
    print(f'{n:20s}', s[LABELS].sum().to_dict(), '| changed vs base:',
          int((s[LABELS].values.argmax(1) != SUBS['base'][LABELS].values.argmax(1)).sum()))

[114.7 min] paper clusters: 7922 | cross-cluster groups with >1 test pair: 131 (285 pairs) | pairs inside one cluster: 2679
base                 {'same_figure': 818, 'same_paper': 2359, 'related_papers': 2877, 'unrelated_papers': 3946} | changed vs base: 0
consistency          {'same_figure': 818, 'same_paper': 2347, 'related_papers': 2892, 'unrelated_papers': 3943} | changed vs base: 70
consistency_within   {'same_figure': 829, 'same_paper': 2398, 'related_papers': 2836, 'unrelated_papers': 3937} | changed vs base: 132


In [12]:
# ===== CELL 12 : submissions + report =====
# submission.csv        -> the safe one (same recipe as the 0.57485 run, plus the new features)
# submission_consistency.csv / submission_consistency_within.csv -> transductive post-processing,
# unverifiable offline (train's graph is too sparse), so submit the base first and A/B these on the LB.
SUBS['base'].to_csv(f'{OUT}/submission.csv', index=False)
SUBS['consistency'].to_csv(f'{OUT}/submission_consistency.csv', index=False)
SUBS['consistency_within'].to_csv(f'{OUT}/submission_consistency_within.csv', index=False)
np.save(f'{OUT}/D_test_prob_all.npy', PROB)

REPORT['spaces'] = sorted(EMB)
REPORT['ocr'] = bool(OCR_OK)
REPORT['vlm'] = bool(VLM_OK)
REPORT['style'] = bool(STYLE_OK)
REPORT['hf_corpus'] = bool(HF_OK)
REPORT['submissions'] = {n: {c: int(s[c].sum()) for c in LABELS} for n, s in SUBS.items()}
json.dump(REPORT, open(f'{OUT}/report_planD.json', 'w'), indent=1, default=str)
print(json.dumps(REPORT, indent=1, default=str))
tlog('done ->', f'{OUT}/submission.csv')

{
 "image+text": {
  "features_scalar": 87,
  "features_all": 5463,
  "lgb_scalar": 0.5153,
  "lgb_all": 0.5007,
  "lr_all": 0.4329,
  "blend_weights": {
   "lgb_scalar": 0.6,
   "lgb_all": 0.3,
   "lr_all": 0.1
  },
  "blend": 0.5238,
  "final": 0.5303
 },
 "text+text": {
  "features_scalar": 59,
  "features_all": 5179,
  "lgb_scalar": 0.6582,
  "lgb_all": 0.6521,
  "lr_all": 0.5825,
  "hf_aug": 0.652,
  "blend_weights": {
   "lgb_scalar": 0.5,
   "lgb_all": 0.0,
   "lr_all": 0.1,
   "hf_aug": 0.4
  },
  "blend": 0.6737,
  "final": 0.6741
 },
 "image+image": {
  "features_scalar": 130,
  "features_all": 7042,
  "lgb_scalar": 0.4975,
  "lgb_all": 0.4997,
  "lr_all": 0.4616,
  "hf_aug": 0.4303,
  "blend_weights": {
   "lgb_scalar": 0.2,
   "lgb_all": 0.3,
   "lr_all": 0.2,
   "hf_aug": 0.3
  },
  "blend": 0.513,
  "final": 0.5163
 },
 "overall_oof_macro_f1": 0.5692,
 "spaces": [
  "clip_i",
  "clip_ic",
  "clip_t",
  "clip_tc",
  "dino_i",
  "qwen_cap",
  "qwen_capc",
  "qwen_hf",
  "qw